In [ ]:
import os
os.listdir()
!pip show streamlit

In [ ]:
# ==========================================
# ENSEMBLE LEARNING PROJECT (COLAB VERSION)
# ==========================================

# ----------- Install (if needed) ----------
!pip install scikit-learn joblib

# ----------- Import Libraries -------------
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report

import joblib

# ----------- Load Dataset -----------------
# Upload your file in Colab first (data.csv or train.csv)
df = pd.read_csv("top_rated_tv.csv")   # change name if needed

print("Dataset Preview:")
print(df.head())

# ----------- Data Preprocessing -----------
df = df.dropna()              # remove missing values

# Separate target variable 'is_highly_rated' based on vote_average
# Assuming 'highly rated' is defined as vote_average >= 7.5
y = (df['vote_average'] >= 7.5).astype(int) # Convert boolean to int (0 or 1) for classification

# Select numerical features for X
X = df[['popularity', 'vote_average', 'vote_count']]

print("\nShape of X:", X.shape)
print("Shape of y (is_highly_rated):", y.shape)

# ----------- Train-Test Split -------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------- Feature Scaling --------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ----------- Random Forest ----------------
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("\n===== Random Forest =====")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# ----------- Decision Tree ----------------
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("\n===== Decision Tree =====")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))

# ----------- Logistic Regression ----------
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))

# ----------- Save Model -------------------
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("\n✅ Model and scaler saved!")

# ----------- Download Files ---------------
from google.colab import files
files.download("rf_model.pkl")
files.download("scaler.pkl")

In [ ]:
# Start a new thread for Streamlit and run ngrok
from pyngrok import ngrok
import subprocess
import threading
import time

def run_streamlit():
    # Save the Streamlit app code to a temporary file
    with open('streamlit_app.py', 'w') as f:
        f.write(
            """import streamlit as st
import numpy as np
import joblib

# Load saved files
model = joblib.load("rf_model.pkl")
scaler = joblib.load("scaler.pkl")

st.title("🌸 Ensemble Learning Prediction App")

st.write("Enter feature values:")

# Change number of inputs based on your dataset
popularity = st.number_input("Popularity", value=0.0)
vote_average = st.number_input("Vote Average", value=0.0)
vote_count = st.number_input("Vote Count", value=0)

if st.button("Predict"):
    # Ensure order matches training features: popularity, vote_average, vote_count
    data = np.array([[popularity, vote_average, vote_count]])
    data = scaler.transform(data)

    prediction = model.predict(data)

    # Updated prediction message for 'is_highly_rated' target
    st.success(f"Prediction: {'Highly Rated' if prediction[0] else 'Not Highly Rated'}")""")

    # Run Streamlit in a subprocess
    print("Starting Streamlit...")
    process = subprocess.Popen(["streamlit", "run", "streamlit_app.py"])
    process.wait() # Wait for the Streamlit process to finish if it exits

# Start Streamlit in a new thread
streamlit_thread = threading.Thread(target=run_streamlit)
streamlit_thread.start()

# Wait for Streamlit to start up and expose it with ngrok
print("Waiting for Streamlit to be accessible...")

# Give Streamlit a moment to start (adjust if needed)
time.sleep(5)

# Get Streamlit's local port (default is 8501)
streamlit_port = 8501

# Setup ngrok tunnel
try:
    # Make sure ngrok is killed from previous runs
    ngrok.kill()
    public_url = ngrok.connect(streamlit_port, bind_tls=True)
    print(f"Streamlit App URL: {public_url}")
except Exception as e:
    print(f"Error setting up ngrok tunnel: {e}")
    print("You might need to install ngrok or have an authentication token. Visit https://dashboard.ngrok.com/get-started/setup for more info.")